# Resampling of DCLP3

In [64]:
import os
import numpy as np
import pandas as pd
from datetime import datetime

### Read Existing Tables

In [65]:
file_path = "../../data/out/DCLP3 Public Dataset - Release 3 - 2022-08-04/"
file_map = {
    'cgm': 'DCLP3_cgm_history.csv.gz',
    'bolus': 'DCLP3_bolus_event_history.csv.gz',
    'basal': 'DCLP3_basal_event_history.csv.gz',
}

In [66]:
def get_df_from_file(file_path, file_name, parse_datetime=True, usecols=None, sep=',', encoding='utf-8'):
    df = pd.read_csv(file_path + file_name, sep=sep, usecols=usecols, encoding=encoding)
    if parse_datetime:
        df['datetime'] = pd.to_datetime(df['datetime'], unit='s')
        df = df.rename(columns={'datetime': 'date'})
    return df

In [67]:
def get_extended_df(original_df, value_column):
    """
    Get a df where quantities are distributed throughout 5-minute intervals instead of having start- and end dates.
    """
    new_rows = []
    for _, row in original_df.iterrows():
        new_rows.extend(split_duration(row, value_column))
    extended_df = pd.DataFrame(new_rows)
    extended_df.set_index('date', inplace=True)
    return extended_df

def split_duration(row, value_column):
    """
    For features with a duration, we split the values across 5-minute intervals by adding
    new rows for every 5-minute window in duration, and equally split the original quantity across those rows.
    """
    duration = row['end_date'] - row['date']
    rounded_duration = round(duration / pd.Timedelta(minutes=5)) * pd.Timedelta(minutes=5)
    num_intervals = rounded_duration // pd.Timedelta(minutes=5)
    if num_intervals < 1:
        num_intervals = 1
    value_per_interval = row[value_column] / num_intervals
    new_rows = []
    for i in range(int(num_intervals)):
        new_row = {
            'date': row['date'] + pd.Timedelta(minutes=5 * i),
            value_column: value_per_interval,
            'patient_id': row['patient_id'],
        }
        new_rows.append(new_row)
    return new_rows

In [68]:
df_glucose = get_df_from_file(file_path, file_map['cgm'])
df_bolus = get_df_from_file(file_path, file_map['bolus'])
df_basal = get_df_from_file(file_path, file_map['basal'])


### Resample Existing Tables

In [69]:
df_glucose.set_index('date', inplace=True)
df_glucose.head()

,patient_id,cgm
date,,
2018-04-19 15:00:16,10,88
2018-04-19 15:04:30,10,85
2018-04-19 15:09:28,10,81
2018-04-19 15:14:28,10,79
2018-04-19 15:19:28,10,75


In [70]:
df_bolus_orig = df_bolus.copy()
df_bolus['end_date'] = df_bolus['date'] + pd.to_timedelta(df_bolus['delivery_duration'], unit='s')
df_bolus = get_extended_df(df_bolus, 'bolus')
df_bolus

,bolus,patient_id
date,,
2018-04-04 13:09:13,0.10,10
2018-04-04 14:40:51,7.45,10
2018-04-04 18:35:55,11.03,10
2018-04-05 03:17:14,4.12,10
2018-04-05 08:39:31,10.50,10
...,...,...
2018-10-09 00:07:51,1.19,98
2018-10-09 00:10:17,1.20,98
2018-10-09 03:54:19,0.47,98


In [71]:
print(f'New sum after distribution of extended boluses: {df_bolus["bolus"].sum():.2f}, should be: {df_bolus_orig["bolus"].sum():.2f}')

New sum after distribution of extended boluses: 575645.20, should be: 575645.20


In [72]:
df_basal_orig = df_basal.copy()
df_basal.sort_values(by=['patient_id', 'date'], inplace=True)
df_basal.set_index('date', inplace=True)
df_basal

,patient_id,basal_rate
date,,
2018-01-05 12:48:07,3,0.600
2018-01-06 00:04:44,3,0.500
2018-01-06 07:00:10,3,0.600
2018-01-07 00:01:14,3,0.500
2018-01-07 07:01:39,3,0.600
...,...,...
2018-10-24 08:26:21,171,0.950
2018-10-24 08:56:22,171,1.119
2018-10-24 09:01:22,171,3.207


In [73]:
processed_dfs = []
subject_ids = df_glucose['patient_id'].unique()
print("Subjects:", len(subject_ids))
for subject_id in subject_ids:
    df_subject = df_glucose[df_glucose['patient_id'] == subject_id].copy()
    df_subject = df_subject[['cgm']].resample('5min', label='right').mean()
    df_subject['patient_id'] = subject_id
    df_subject.sort_index(inplace=True)

    def merge_data(df_col, df_subject, col_names, subject_id, agg_type='sum'):
        """ agg_type is data aggregation type. """
        df_subset = df_col[df_col['patient_id'] == subject_id].copy()
        if not df_subset.empty:
            if agg_type == 'mean':
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').mean()
            elif agg_type == 'sum':
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').sum()
            elif agg_type == 'first':
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').first()
            elif agg_type == 'last':
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').last()
            else:
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').sum()
            df_subject = pd.merge(df_subject, df_subset, on="date", how='outer')
        else:
            df_subject[col_names] = np.nan
        return df_subject

    # Add insulin and insulin type
    df_subject = merge_data(df_bolus, df_subject, ['bolus'], subject_id, agg_type='sum')
    df_subject = merge_data(df_basal, df_subject, ['basal_rate'], subject_id, agg_type='last')
    df_subject['basal_rate'] = df_subject['basal_rate'].ffill()
    
    df_subject['patient_id'] = subject_id
    df_subject = df_subject.rename(columns={'patient_id': 'id', 'basal_rate': 'basal', 'cgm': 'CGM'})
    processed_dfs.append(df_subject)
    print(f"{subject_id} is finished processing")

df_final = pd.concat(processed_dfs)

Subjects: 112
10 is finished processing
100 is finished processing
102 is finished processing
105 is finished processing
111 is finished processing
114 is finished processing
115 is finished processing
116 is finished processing
117 is finished processing
118 is finished processing
120 is finished processing
121 is finished processing
122 is finished processing
123 is finished processing
126 is finished processing
128 is finished processing
129 is finished processing
13 is finished processing
130 is finished processing
132 is finished processing
133 is finished processing
134 is finished processing
135 is finished processing
137 is finished processing
138 is finished processing
14 is finished processing
141 is finished processing
142 is finished processing
143 is finished processing
144 is finished processing
145 is finished processing
147 is finished processing
148 is finished processing
149 is finished processing
15 is finished processing
150 is finished processing
153 is finished pr

In [74]:
df_final

,CGM,id,bolus,basal
date,,,,
2018-04-04 12:55:00,NaN,10,NaN,2.0
2018-04-04 13:00:00,NaN,10,NaN,0.0
2018-04-04 13:05:00,NaN,10,NaN,0.0
2018-04-04 13:10:00,NaN,10,0.1,0.0
2018-04-04 13:15:00,NaN,10,0.0,0.0
...,...,...,...,...
2018-10-09 11:40:00,133.0,98,NaN,0.6
2018-10-09 11:45:00,137.0,98,NaN,0.6
2018-10-09 11:50:00,139.0,98,NaN,0.6


### Add Additional Tables

We found:
- Carbs
- Insulin type
- Age
- Weight
- Height
- Gender

In [75]:
raw_data_file_path = "../../data/raw/DCLP3 Public Dataset - Release 3 - 2022-08-04/Data Files/"

In [76]:
# Adding carbs
df_carbs = get_df_from_file(raw_data_file_path, 'RocheMeter_a.txt', parse_datetime=False, sep='|', encoding='utf-16')
df_carbs['date'] = pd.to_datetime(df_carbs.DataDtTm_adjusted.fillna(df_carbs.DataDtTm))
#df_carbs = df_carbs[df_carbs['Carbs'] > 0][['PtID', 'date', 'Carbs']]
#df_carbs = df_carbs.rename(columns={'PtId': 'id', 'CarbInput': 'carbs'})
df_carbs.head()

,PtID,RecID,DataDtTm,BG,Carbs,IsQCTest,SystemDefinedEvents,UserDefinedEvents,Flags,IsQCtestJaeb,DataDtTm_adjusted,date
0,9,988,2018-02-01 12:38:48,282,NaN,True,Control,NaN,NaN,NaN,NaN,2018-02-01 12:38:48
1,9,989,2018-02-01 12:38:19,44,NaN,True,Control,NaN,NaN,NaN,NaN,2018-02-01 12:38:19
2,9,990,2018-01-10 20:29:44,93,NaN,False,NaN,NaN,NaN,NaN,NaN,2018-01-10 20:29:44
3,9,991,2018-01-09 18:49:31,256,NaN,False,NaN,NaN,NaN,NaN,NaN,2018-01-09 18:49:31
4,9,992,2018-01-08 10:26:08,238,NaN,False,NaN,NaN,NaN,NaN,NaN,2018-01-08 10:26:08


In [77]:
# We observe that DCLP3 has no carb events
df_carbs[df_carbs['Carbs'].notna()]

,PtID,RecID,DataDtTm,BG,Carbs,IsQCTest,SystemDefinedEvents,UserDefinedEvents,Flags,IsQCtestJaeb,DataDtTm_adjusted,date


In [79]:
df_insulin_type = get_df_from_file(raw_data_file_path, 'Insulin_a.txt', parse_datetime=False, sep='|', encoding='utf-16')
df_insulin_type = df_insulin_type[['PtID', 'ParentInsulinListID']]
df_insulin_type = df_insulin_type.rename(columns={'PtID': 'id', 'ParentInsulinListID': 'insulin_type'})
df_insulin_type

,id,insulin_type
0,9,Novolog (Aspart)
1,10,Lantus (Glargine) 2 times per day
2,10,Humalog (Lispro)
3,10,Humalog (Lispro)
4,10,Novolog (Aspart)
...,...,...
250,166,Humalog (Lispro)
251,167,Novolog (Aspart)
252,169,Humalog (Lispro)
253,170,Novolog (Aspart)


In [80]:
def add_single_value_to_subjects(df, df_new_val, col_name):
    # TODO: this function is very inefficient... add value directly to located rows instead
    processed_dfs = []
    subject_ids = df['id'].unique()
    for subject_id in subject_ids:
        df_subject = df[df['id'] == subject_id].copy()
        df_subject.sort_index(inplace=True)
    
        user_data = df_new_val[df_new_val['id'] == subject_id].copy()
        if not user_data.empty:
            df_subject[col_name] = user_data[col_name].iloc[0]
        else:
            df_subject[col_name] = np.nan        
        processed_dfs.append(df_subject)
        
    df = pd.concat(processed_dfs)
    return df

In [81]:
# Add insulin type
df_final = add_single_value_to_subjects(df_final, df_insulin_type, 'insulin_type')
df_final

,CGM,id,bolus,basal,insulin_type
date,,,,,
2018-04-04 12:55:00,NaN,10,NaN,2.0,Lantus (Glargine) 2 times per day
2018-04-04 13:00:00,NaN,10,NaN,0.0,Lantus (Glargine) 2 times per day
2018-04-04 13:05:00,NaN,10,NaN,0.0,Lantus (Glargine) 2 times per day
2018-04-04 13:10:00,NaN,10,0.1,0.0,Lantus (Glargine) 2 times per day
2018-04-04 13:15:00,NaN,10,0.0,0.0,Lantus (Glargine) 2 times per day
...,...,...,...,...,...
2018-10-09 11:40:00,133.0,98,NaN,0.6,Novolog (Aspart)
2018-10-09 11:45:00,137.0,98,NaN,0.6,Novolog (Aspart)
2018-10-09 11:50:00,139.0,98,NaN,0.6,Novolog (Aspart)


In [82]:
# Add age and gender
df_user_data = get_df_from_file(raw_data_file_path, 'DiabScreening_a.txt', parse_datetime=False, sep='|', encoding='utf-16')
df_user_data = df_user_data[['PtID', 'Gender', 'AgeAtEnrollment']]
df_user_data = df_user_data.rename(columns={'PtID': 'id', 'Gender': 'gender', 'AgeAtEnrollment': 'age'})
df_user_data

,id,gender,age
0,9,M,23
1,10,F,38
2,12,F,37
3,14,F,45
4,15,F,35
...,...,...,...
165,166,F,14
166,167,F,29
167,169,M,44
168,170,F,14


In [91]:
# Check inhale data
df_user_data = get_df_from_file(raw_data_file_path, 'DiabScreening_a.txt', parse_datetime=False, sep='|', encoding='utf-16')
df_user_data['InsModInhaled'].unique()


array([nan])

In [83]:
# Add age and gender
df_final = add_single_value_to_subjects(df_final, df_user_data, 'gender')
df_final = add_single_value_to_subjects(df_final, df_user_data, 'age')
df_final.head()

,CGM,id,bolus,basal,insulin_type,gender,age
date,,,,,,,
2018-04-04 12:55:00,NaN,10,NaN,2.0,Lantus (Glargine) 2 times per day,F,38
2018-04-04 13:00:00,NaN,10,NaN,0.0,Lantus (Glargine) 2 times per day,F,38
2018-04-04 13:05:00,NaN,10,NaN,0.0,Lantus (Glargine) 2 times per day,F,38
2018-04-04 13:10:00,NaN,10,0.1,0.0,Lantus (Glargine) 2 times per day,F,38
2018-04-04 13:15:00,NaN,10,0.0,0.0,Lantus (Glargine) 2 times per day,F,38


In [84]:
df_weight = get_df_from_file(raw_data_file_path, 'AdvEvent_a.txt', parse_datetime=False, sep='|',
                                encoding='utf-16')
#df_weight = df_weight[['PtID', 'Gender', 'AgeAtEnrollment']]
#df_weight = df_weight.rename(columns={'PtID': 'id', 'Gender': 'gender', 'AgeAtEnrollment': 'age'})
df_weight[['PtID', 'Weight']]
# Too few samples, looking in another table...

,PtID,Weight
0,14,NaN
1,29,NaN
2,88,NaN
3,155,NaN
4,115,NaN
5,122,NaN
6,164,NaN
7,26,65.0
8,45,NaN
9,60,NaN


In [85]:
df_weight_and_height = get_df_from_file(raw_data_file_path, 'DiabPhysExam_a.txt', parse_datetime=False, sep='|',
                                encoding='utf-16')
df_weight_and_height = df_weight_and_height[['PtID', 'Weight', 'Height']]
df_weight_and_height = df_weight_and_height.rename(columns={'PtID': 'id', 'Weight': 'weight', 'Height': 'height'})
df_weight_and_height

,id,weight,height
0,9,177.0,177.0
1,10,237.0,169.0
2,12,161.0,170.0
3,14,249.0,165.0
4,15,108.0,176.5
...,...,...,...
165,166,138.0,171.5
166,167,61.1,64.0
167,169,78.0,70.0
168,170,62.6,167.4


In [86]:
# Add weight and height
df_final = add_single_value_to_subjects(df_final, df_weight_and_height, 'weight')
df_final = add_single_value_to_subjects(df_final, df_weight_and_height, 'height')
df_final.head()

,CGM,id,bolus,basal,insulin_type,gender,age,weight,height
date,,,,,,,,,
2018-04-04 12:55:00,NaN,10,NaN,2.0,Lantus (Glargine) 2 times per day,F,38,237.0,169.0
2018-04-04 13:00:00,NaN,10,NaN,0.0,Lantus (Glargine) 2 times per day,F,38,237.0,169.0
2018-04-04 13:05:00,NaN,10,NaN,0.0,Lantus (Glargine) 2 times per day,F,38,237.0,169.0
2018-04-04 13:10:00,NaN,10,0.1,0.0,Lantus (Glargine) 2 times per day,F,38,237.0,169.0
2018-04-04 13:15:00,NaN,10,0.0,0.0,Lantus (Glargine) 2 times per day,F,38,237.0,169.0


### Save Resampled Data

In [87]:
df_final

,CGM,id,bolus,basal,insulin_type,gender,age,weight,height
date,,,,,,,,,
2018-04-04 12:55:00,NaN,10,NaN,2.0,Lantus (Glargine) 2 times per day,F,38,237.0,169.0
2018-04-04 13:00:00,NaN,10,NaN,0.0,Lantus (Glargine) 2 times per day,F,38,237.0,169.0
2018-04-04 13:05:00,NaN,10,NaN,0.0,Lantus (Glargine) 2 times per day,F,38,237.0,169.0
2018-04-04 13:10:00,NaN,10,0.1,0.0,Lantus (Glargine) 2 times per day,F,38,237.0,169.0
2018-04-04 13:15:00,NaN,10,0.0,0.0,Lantus (Glargine) 2 times per day,F,38,237.0,169.0
...,...,...,...,...,...,...,...,...,...
2018-10-09 11:40:00,133.0,98,NaN,0.6,Novolog (Aspart),F,26,142.0,166.0
2018-10-09 11:45:00,137.0,98,NaN,0.6,Novolog (Aspart),F,26,142.0,166.0
2018-10-09 11:50:00,139.0,98,NaN,0.6,Novolog (Aspart),F,26,142.0,166.0


In [88]:
save_file_path = "../../data/resampled/"
os.makedirs(save_file_path, exist_ok=True)
df_final.to_csv(save_file_path + 'DCLP3.csv')